# __Data Cleaning Appendix__

Importing and Environment Setup: We import core data processing libraries and set view options to inspect all dataset columns during data cleaning.

In [3]:
# Code Cell 1: imports and settings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import duckdb, sqlalchemy
pd.set_option("display.max_columns", None)

Loading Raw Datasets: We load our three raw CSV files into pandas DataFrames and print their initial dimensions.

In [4]:
# Code Cell 2: Ingest/Load CSV files into DataFrames
cmu_df = pd.read_csv('cmu-sleep.csv')
fitbit_df = pd.read_csv('FitbitSleep(1-30-20).csv')
grades_df = pd.read_csv('CourseGrades(3-6-20).csv')

# Display raw dataset dimensions
print("Raw CMU Dataset Dimensions:", cmu_df.shape)
print("Raw Fitbit Logs Dimensions:", fitbit_df.shape)
print("Raw Course Grades Dimensions:", grades_df.shape)

Raw CMU Dataset Dimensions: (634, 15)
Raw Fitbit Logs Dimensions: (333708, 10)
Raw Course Grades Dimensions: (11070, 4)


Ingestion Baseline/Intermediate Checkpoint 1: We save local copies of the raw ingested DataFrames as an intermediate checkpoint to document our raw baseline state before performing variable cleaning and transformations.

In [5]:
# Code Cell 3: Export Intermediate Checkpoint 1
cmu_df.to_csv("checkpoint_01_raw_cmu.csv", index=False)
fitbit_df.to_csv("checkpoint_01_raw_fitbit.csv", index=False)
grades_df.to_csv("checkpoint_01_raw_grades.csv", index=False)

print("Checkpoint 1 files saved successfully!")

Checkpoint 1 files saved successfully!


Cleaning Course Performance Data/Data Transformation 1: We drop records missing final letter grades (`FinalGrade`), map letter grades (A through F) to standard 4.0 numerical scale values (`numeric_grade`), and compute each student's average course performance (`nethealth_course_gpa`) grouped by `egoid`.

In [6]:
# Code Cell 4: Clean letter grades and calculate student course GPA
# Drop rows with missing final letter grades
clean_grades_df = grades_df.dropna(subset=["FinalGrade"]).copy()

# Map letter grades to standard 4.0 GPA numerical scale
letter_grade_map = {
    "A": 4.0,
    "A-": 3.7,
    "B+": 3.3,
    "B": 3.0,
    "B-": 2.7,
    "C+": 2.3,
    "C": 2.0,
    "C-": 1.7,
    "D+": 1.3,
    "D": 1.0,
    "F": 0.0,
}
clean_grades_df["numeric_grade"] = clean_grades_df["FinalGrade"].map(
    letter_grade_map
)

# Group by student ID (egoid) and aggregate average grade across courses
student_gpa_df = (
    clean_grades_df.groupby("egoid")["numeric_grade"].mean().reset_index()
)
student_gpa_df.rename(
    columns={"numeric_grade": "nethealth_course_gpa"}, inplace=True
)

# Inspect cleaned GPA summary
print("Cleaned Student GPA Summary Shape:", student_gpa_df.shape)
student_gpa_df.head()

Cleaned Student GPA Summary Shape: (220, 2)


,egoid,nethealth_course_gpa
0,10237,3.487805
1,11002,3.880000
2,13486,3.493878
3,14279,3.848936
4,14301,3.750000


Cleaning Fitbit Actigraphy Logs/Data Transformation 2: We filter out daily tracking logs missing `minsasleep`, convert sleep duration from minutes to hours (`avg_daily_sleep_hours`), and aggregate student-level biometric metrics: average daily sleep hours, average sleep efficiency (`avg_sleep_efficiency`), and average nighttime awakenings (`avg_mins_awake`).

In [7]:
# Code Cell 5: Clean daily Fitbit sleep logs and compute summary metrics
# Filter out missing sleep duration entries
clean_fitbit_df = fitbit_df.dropna(subset=["minsasleep"]).copy()

# Convert minutes asleep to total sleep hours
clean_fitbit_df["sleep_hours"] = clean_fitbit_df["minsasleep"] / 60.0

# Aggregate daily Fitbit logs into student-level averages
student_sleep_summary = (
    clean_fitbit_df.groupby("egoid")
    .agg(
        avg_daily_sleep_hours=("sleep_hours", "mean"),
        avg_sleep_efficiency=("Efficiency", "mean"),
        avg_mins_awake=("minsawake", "mean"),
    )
    .reset_index()
)

# Inspect clean sleep summary
print("Cleaned Student Sleep Summary Shape:", student_sleep_summary.shape)
student_sleep_summary.head()

Cleaned Student Sleep Summary Shape: (622, 4)


,egoid,avg_daily_sleep_hours,avg_sleep_efficiency,avg_mins_awake
0,10237,6.334210,0.962233,14.789679
1,10469,4.659869,0.932549,20.290196
2,10547,5.470370,0.918469,29.629630
3,11002,6.468560,0.950600,20.230679
4,11128,5.675787,0.930784,26.555556


Processed Sub-Tables/Intermediate Checkpoint 2: We export clean intermediate CSV files for both processed sub-tables (`checkpoint_02_student_gpa_clean.csv` and `checkpoint_02_student_sleep_clean.csv`) before merging them into our final dataset.

In [8]:
# Code Cell 6: Export Intermediate Checkpoint 2
student_gpa_df.to_csv("checkpoint_02_student_gpa_clean.csv", index=False)
student_sleep_summary.to_csv(
    "checkpoint_02_student_sleep_clean.csv", index=False
)

print("Checkpoint 2 files saved successfully!")

Checkpoint 2 files saved successfully!


Before merging the CMU sleep dataset with the Notre Dame NetHealth dataset, we need to assess whether the CMU `subject_id` and Notre Dame `egoid` identifiers correspond to the same individuals. The CMU dataset includes five study cohorts, including the Notre Dame cohort (`study = 4`, `cohort = nh`), so we compare the unique participant IDs across both datasets and examine which CMU cohorts contain matching IDs. This helps us determine whether merging the datasets by participant ID is justfied, rather than concatenating them as separate observations.

In [9]:
# Code Cell 7: Compare the Identifiers

# Check whether participant identifiers overlap across datasets
cmu_ids = set(cmu_df["subject_id"].dropna())
nd_ids = set(student_sleep_summary["egoid"].dropna())

matching_ids = cmu_ids & nd_ids

print("Unique CMU participants:", len(cmu_ids))
print("Unique Notre Dame participants:", len(nd_ids))
print("Number of matching IDs:", len(matching_ids))
print("Percentage of Notre Dame IDs found in CMU:",
      round(len(matching_ids) / len(nd_ids) * 100, 2), "%")

# Check which CMU study/cohort contains the matching IDs.
print("\nMatching IDs by CMU study and cohort:")

display(
    cmu_df[cmu_df["subject_id"].isin(matching_ids)]
    .groupby(["study", "cohort"])
    .agg(
        unique_participants=("subject_id", "nunique")
    )
    .reset_index()
)

Unique CMU participants: 547
Unique Notre Dame participants: 622
Number of matching IDs: 147
Percentage of Notre Dame IDs found in CMU: 23.63 %

Matching IDs by CMU study and cohort:


,study,cohort,unique_participants
0,4,nh,147


In [10]:
# Code Cell 8: Inspect the Matching Records

# Display examples of records with matching participant IDs
cmu_overlap = cmu_df[
    cmu_df["subject_id"].isin(matching_ids)
].copy()

nd_overlap = student_sleep_summary[
    student_sleep_summary["egoid"].isin(matching_ids)
].copy()

print("Matching unique CMU IDs:",
      cmu_overlap["subject_id"].nunique())

print("Matching unique Notre Dame IDs:",
      nd_overlap["egoid"].nunique())

print("Same set of participant IDs:",
      set(cmu_overlap["subject_id"].dropna()) ==
      set(nd_overlap["egoid"].dropna()))

Matching unique CMU IDs: 147
Matching unique Notre Dame IDs: 147
Same set of participant IDs: True


We identified 147 participant IDs shared between the CMU and Notre Dame sleep datasets. All 147 matching IDs belong to CMU Study 4 (nh), while no matches were found in the other four CMU cohorts. This pattern is consistent with the expected overlap between the data and these results support merging the datasets by participant IDs for the overlapping participants.

Joining Datasets on Shared Student Keys/Data Transformation 3: We execute an inner join between the student Fitbit sleep summary and the student course GPA table on `egoid`[cite: 1, 5]. We align key names by renaming `egoid` to `subject_id` and merge with the multi-cohort CMU sleep dataset (`cmu-sleep.csv`) using `subject_id` to build our final analysis-ready dataset.

In [11]:
# Code Cell 9: Execute legitimate joins on shared student keys
# Inner join NetHealth sleep summary and course GPA on egoid
nethealth_merged_df = pd.merge(
    student_sleep_summary, student_gpa_df, on="egoid", how="inner"
)

# Standardize key column name across datasets
nethealth_merged_df.rename(columns={"egoid": "subject_id"}, inplace=True)

# Inner join NetHealth metrics with CMU master dataset on subject_id
analysis_ready_df = pd.merge(
    cmu_df, nethealth_merged_df, on="subject_id", how="inner"
)

# Verify final dimensions and inspect structure
print("Analysis-Ready Master Dataset Shape:", analysis_ready_df.shape)
analysis_ready_df.head()

Analysis-Ready Master Dataset Shape: (147, 19)


,subject_id,study,cohort,demo_race,demo_gender,demo_firstgen,bedtime_mssd,TotalSleepTime,midpoint_sleep,frac_nights_with_data,daytime_sleep,cum_gpa,term_gpa,term_units,Zterm_units_ZofZ,avg_daily_sleep_hours,avg_sleep_efficiency,avg_mins_awake,nethealth_course_gpa
0,14847,4,nh,1,1,1,0.131813,456.566667,385.433333,1.000000,51.033333,3.333167,3.666833,,,5.891124,0.947670,19.722074,3.437500
1,17020,4,nh,0,0,1,0.209352,429.607143,425.339286,0.933333,59.678571,3.166667,3.500000,,,6.163119,0.941621,22.333028,3.370000
2,18707,4,nh,1,1,1,0.052274,419.900000,310.900000,1.000000,57.000000,2.933400,3.277833,,,6.109016,0.963410,12.960452,3.371795
3,51723,4,nh,1,0,1,0.265969,273.916667,455.750000,0.800000,30.500000,2.800200,3.466800,,,4.822786,0.960292,12.600000,3.371795
4,72885,4,nh,1,0,1,0.083406,356.833333,414.416667,1.000000,45.766667,3.333400,3.266600,,,5.535082,0.920509,29.466618,3.270000


Save Final Analysis-Ready Dataset/Intermediate Checkpoint 3: We export the final merged DataFrame to `final_merged_sleep_gpa.csv` so our team members can load analysis-ready data directly for exploratory data analysis and preregistration without re-running raw data cleaning.

In [12]:
# Code Cell 10: Export Final Analysis-Ready Checkpoint 3
analysis_ready_df.to_csv("final_merged_sleep_gpa.csv", index=False)

print(
    "Final analysis-ready dataset saved successfully to final_merged_sleep_gpa.csv!"
)

Final analysis-ready dataset saved successfully to final_merged_sleep_gpa.csv!
